# В даннном ноутбуке проходит первая часть создания МЕТА модели, а именно получение логитов каждой базовой модели, для дальнейшего обучения МЕТА модели

# Обучение финальной МЕТА нейросети

# Подготовка данных

# Разбиваем на тренировочную и валидационную подвыборки

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Загружаем полную тренировочную выборку
print(" Загружаем полную тренировочную выборку...")
full_train_df = pd.read_csv('3/train_data.csv')
print(f" Загружено {len(full_train_df)} записей")

# Разделяем на тренировочную и валидационную выборки для мета-нейросети
print("\n Разделение данных для мета-нейросети (85%/15%)...")
meta_train_df, meta_val_df = train_test_split(
    full_train_df,
    test_size=0.1765,
    random_state=33,
    stratify=full_train_df['No Finding']  # балансируем по классу "No Finding"
)

print(f" Разделение завершено:")
print(f"   • Мета-train: {len(meta_train_df)} записей")
print(f"   • Мета-val: {len(meta_val_df)} записей")

# Проверяем балансировку
print(f"\n Балансировка по 'No Finding':")
print(f"   Мета-train: {(meta_train_df['No Finding'] == 1).sum()} записей ({(meta_train_df['No Finding'] == 1).sum()/len(meta_train_df)*100:.1f}%)")
print(f"   Мета-val: {(meta_val_df['No Finding'] == 1).sum()} записей ({(meta_val_df['No Finding'] == 1).sum()/len(meta_val_df)*100:.1f}%)")

# Сохраняем
print("\n Сохраняем...")
meta_train_df.to_csv('meta_train_data.csv', index=False)
meta_val_df.to_csv('meta_val_data.csv', index=False)
print(" Готово")

 Загружаем полную тренировочную выборку...
 Загружено 95302 записей

 Разделение данных для мета-нейросети (85%/15%)...
 Разделение завершено:
   • Мета-train: 78481 записей
   • Мета-val: 16821 записей

 Балансировка по 'No Finding':
   Мета-train: 42251 записей (53.8%)
   Мета-val: 9056 записей (53.8%)

 Сохраняем...
 Готово


# Полный цикл получения логитов от модели DenseNet-121

# создаем класс ChestXrayDataset

In [7]:
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['full_path']
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, idx  # Возвращаем индекс для сопоставления

print(" Класс ChestXrayDataset создан")

 Класс ChestXrayDataset создан


# создаем трансформации для модели DenseNet-121

In [8]:
from torchvision import transforms

# Трансформации для DenseNet-121 (такие же как при обучении)
densenet_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # DenseNet-121 обучался на 224x224
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print(" Трансформации для DenseNet-121 созданы")
print(f"   Размер: 224×224")
print(f"   Нормализация: mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]")

 Трансформации для DenseNet-121 созданы
   Размер: 224×224
   Нормализация: mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]


# Создаем dataset для DenseNet-121

In [9]:
import pandas as pd
import torch

# Загружаем мета-тренировочные данные
meta_train_df = pd.read_csv('meta_train_data.csv')
meta_val_df = pd.read_csv('meta_val_data.csv')

print(" Загружены мета-выборки:")
print(f"   • Мета-train: {len(meta_train_df)} изображений")
print(f"   • Мета-val: {len(meta_val_df)} изображений")

# Создаем Dataset объекты
meta_train_dataset = ChestXrayDataset(meta_train_df, transform=densenet_transform)
meta_val_dataset = ChestXrayDataset(meta_val_df, transform=densenet_transform)

print(" Dataset объекты созданы")
print(f"   • Train dataset: {len(meta_train_dataset)} элементов")
print(f"   • Val dataset: {len(meta_val_dataset)} элементов")

# Сохраняем датасеты
torch.save(meta_train_dataset, 'meta_train_dataset_densenet.pth')
torch.save(meta_val_dataset, 'meta_val_dataset_densenet.pth')

print(" Dataset сохранены:")
print(f"   • meta_train_dataset_densenet.pth")
print(f"   • meta_val_dataset_densenet.pth")

 Загружены мета-выборки:
   • Мета-train: 78481 изображений
   • Мета-val: 16821 изображений
 Dataset объекты созданы
   • Train dataset: 78481 элементов
   • Val dataset: 16821 элементов
 Dataset сохранены:
   • meta_train_dataset_densenet.pth
   • meta_val_dataset_densenet.pth


# Создание dataloader

In [10]:
from torch.utils.data import DataLoader

batch_size = 32

meta_train_loader = DataLoader(
    meta_train_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

meta_val_loader = DataLoader(
    meta_val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

print(" DataLoader созданы")
print(f"    Batch size: {batch_size}")
print(f"    Train batches: {len(meta_train_loader)}")
print(f"    Val batches: {len(meta_val_loader)}")

 DataLoader созданы
   • Batch size: 32
   • Train batches: 2453
   • Val batches: 526


# загружаем модель DenseNet-121

In [13]:
#  Определяем diseases_list
diseases_list = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 
                 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration',
                 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening',
                 'Pneumonia', 'Pneumothorax']

# 5. Загружаем модель если еще не загружена
print("\n Загружаем модель...")
import torchvision.models.densenet
torch.serialization.add_safe_globals([torchvision.models.densenet.DenseNet])

# Укажи правильный путь к файлу модели
model_path = 'full_model_dn_121.pth'  # или 'best_model.pth'
modelDN_121 = torch.load(model_path, weights_only=False)

# 6. Определяем устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Устройство: {device}")

# 7. Перемещаем модель на устройство
modelDN_121 = model.to(device)
modelDN_121.eval()


🧠 Загружаем модель...
📱 Устройство: cuda


DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

# Получение логитов модели DenseNet-121 

In [14]:
import torch
import numpy as np
from tqdm.notebook import tqdm

print(" Начинаем получение логитов от DenseNet-121...")

# Списки для хранения результатов
train_logits_dn121 = []
train_indices_dn121 = []
val_logits_dn121 = []
val_indices_dn121 = []

# Функция для получения логитов из DataLoader
def get_logits_from_model(model, dataloader, device):
    model.eval()
    all_logits = []
    all_indices = []
    
    with torch.no_grad():
        for batch_images, batch_indices in tqdm(dataloader, desc="Обработка батчей"):
            batch_images = batch_images.to(device)
            batch_logits = model(batch_images)
            
            all_logits.append(batch_logits.cpu().numpy())
            all_indices.append(batch_indices.numpy())
    
    # Объединяем все батчи
    all_logits = np.vstack(all_logits)
    all_indices = np.concatenate(all_indices)
    
    return all_logits, all_indices

# Получаем логиты для тренировочной выборки
print("\n Получаем логиты для тренировочной выборки...")
train_logits_dn121, train_indices_dn121 = get_logits_from_model(
    modelDN_121, meta_train_loader, device
)
print(f" Получено {len(train_logits_dn121)} логитов")
print(f"   Размер: {train_logits_dn121.shape}")

# Получаем логиты для валидационной выборки
print("\n Получаем логиты для валидационной выборки...")
val_logits_dn121, val_indices_dn121 = get_logits_from_model(
    modelDN_121, meta_val_loader, device
)
print(f" Получено {len(val_logits_dn121)} логитов")
print(f"   Размер: {val_logits_dn121.shape}")

# Проверяем соответствие индексов
print(f"\n Проверка соответствия индексов:")
print(f"   Train: индексы от {train_indices_dn121.min()} до {train_indices_dn121.max()}")
print(f"   Val: индексы от {val_indices_dn121.min()} до {val_indices_dn121.max()}")

# Сохраняем логиты в файл
print("\n Сохраняем логиты DenseNet-121...")
np.savez_compressed(
    'densenet121_logits.npz',
    train_logits=train_logits_dn121,
    train_indices=train_indices_dn121,
    val_logits=val_logits_dn121,
    val_indices=val_indices_dn121
)

print(" Логиты сохранены в 'densenet121_logits.npz'")
print(f"\n Сводка:")
print(f"   • Train: {train_logits_dn121.shape} (логиты × классы)")
print(f"   • Val: {val_logits_dn121.shape} (логиты × классы)")
print(f"   • Всего логитов: {len(train_logits_dn121) + len(val_logits_dn121)}")

# Пример первых нескольких логитов
print(f"\n Пример логитов (первые 3 записи):")
for i in range(min(3, len(train_logits_dn121))):
    print(f"   Запись {i}:")
    print(f"     Min: {train_logits_dn121[i].min():.3f}, Max: {train_logits_dn121[i].max():.3f}")
    print(f"     Mean: {train_logits_dn121[i].mean():.3f}, Std: {train_logits_dn121[i].std():.3f}")

 Начинаем получение логитов от DenseNet-121...

 Получаем логиты для тренировочной выборки...


Обработка батчей:   0%|          | 0/2453 [00:00<?, ?it/s]

 Получено 78481 логитов
   Размер: (78481, 15)

 Получаем логиты для валидационной выборки...


Обработка батчей:   0%|          | 0/526 [00:00<?, ?it/s]

 Получено 16821 логитов
   Размер: (16821, 15)

🔍 Проверка соответствия индексов:
   Train: индексы от 0 до 78480
   Val: индексы от 0 до 16820

 Сохраняем логиты DenseNet-121...
 Логиты сохранены в 'densenet121_logits.npz'

 Сводка:
   • Train: (78481, 15) (логиты × классы)
   • Val: (16821, 15) (логиты × классы)
   • Всего логитов: 95302

 Пример логитов (первые 3 записи):
   Запись 0:
     Min: -1.794, Max: -0.347
     Mean: -0.988, Std: 0.386
   Запись 1:
     Min: -1.834, Max: -0.406
     Mean: -1.014, Std: 0.453
   Запись 2:
     Min: -2.805, Max: -0.222
     Mean: -1.049, Std: 0.622


# загрузка логитов модели DenseNet-121 для проверки

In [5]:
import numpy as np

# Загружаем логиты DenseNet-121
logits_data = np.load('densenet121_logits.npz')

# Записываем в переменные
train_logits_dn121 = logits_data['train_logits']
train_indices_dn121 = logits_data['train_indices']
val_logits_dn121 = logits_data['val_logits']
val_indices_dn121 = logits_data['val_indices']

print(" Логиты загружены в переменные")
print(f"   train_logits_dn121: {train_logits_dn121.shape}")
print(f"   val_logits_dn121: {val_logits_dn121.shape}")

✅ Логиты загружены в переменные
   train_logits_dn121: (78481, 15)
   val_logits_dn121: (16821, 15)


# Полный цикл получения логитов от модели EfficientNetV2-S

# загрузка полной модели EfficientNetV2-S

In [34]:
import torch
import torch.nn as nn
from torchvision import models, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

diseases_list = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 
                 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration',
                 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening',
                 'Pneumonia', 'Pneumothorax']

def get_efficientnetv2_s(num_classes=15):
    model = models.efficientnet_v2_s(pretrained=False)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.4, inplace=True),
        nn.Linear(in_features, 768),
        nn.BatchNorm1d(768),
        nn.SiLU(inplace=True),
        nn.Dropout(0.2, inplace=True),
        nn.Linear(768, 384),
        nn.BatchNorm1d(384),
        nn.SiLU(inplace=True),
        nn.Dropout(0.133, inplace=True),
        nn.Linear(384, num_classes)
    )
    return model

efficientnet_model = get_efficientnetv2_s(num_classes=len(diseases_list))
checkpoint = torch.load('efficientnetv2_s_final.pth', map_location=device)
efficientnet_model.load_state_dict(checkpoint['model_state_dict'])
efficientnet_model = model.to(device)
efficientnet_model.eval()

val_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print(f"Модель загружена на {device}")

Модель загружена на cuda


# Получение логитов модели EfficientNetV2-S

In [35]:
import numpy as np
from tqdm.notebook import tqdm

print(" Получение логитов от EfficientNetV2-S...")

# Создаем Dataset с правильными трансформациями
from torch.utils.data import Dataset
from PIL import Image

class EfficientNetDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['full_path']
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, idx

# Создаем Dataset
meta_train_df = pd.read_csv('meta_train_data.csv')
meta_val_df = pd.read_csv('meta_val_data.csv')

train_dataset_eff = EfficientNetDataset(meta_train_df, transform=val_transform)
val_dataset_eff = EfficientNetDataset(meta_val_df, transform=val_transform)

# Создаем DataLoader
batch_size = 16
train_loader_eff = torch.utils.data.DataLoader(
    train_dataset_eff, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=0
)
val_loader_eff = torch.utils.data.DataLoader(
    val_dataset_eff, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=0
)

print(f" DataLoader созданы:")
print(f"   Train: {len(train_loader_eff)} батчей")
print(f"   Val: {len(val_loader_eff)} батчей")

# Функция получения логитов
def get_logits(model, loader, device):
    model.eval()
    all_logits = []
    all_indices = []
    
    with torch.no_grad():
        for images, indices in tqdm(loader, desc="Получение логитов"):
            images = images.to(device)
            logits = model(images)
            
            all_logits.append(logits.cpu().numpy())
            all_indices.append(indices.numpy())
    
    # Объединяем
    all_logits = np.vstack(all_logits)
    all_indices = np.concatenate(all_indices)
    
    return all_logits, all_indices

# Получаем логиты
print("\n Получаем логиты для тренировочной выборки...")
train_logits_effnet, train_indices_effnet = get_logits(efficientnet_model, train_loader_eff, device)

print("\n Получаем логиты для валидационной выборки...")
val_logits_effnet, val_indices_effnet = get_logits(efficientnet_model, val_loader_eff, device)

print(f"\n Логиты получены:")
print(f"   Train logits: {train_logits_effnet.shape}")
print(f"   Val logits: {val_logits_effnet.shape}")

# Получаем метки в правильном порядке
train_labels_effnet = meta_train_df.iloc[train_indices_effnet][diseases_list].values.astype(np.float32)
val_labels_effnet = meta_val_df.iloc[val_indices_effnet][diseases_list].values.astype(np.float32)

print(f"   Train labels: {train_labels_effnet.shape}")
print(f"   Val labels: {val_labels_effnet.shape}")

# Сохраняем все
print("\n Сохраняем логиты...")
np.savez_compressed(
    'efficientnet_logits.npz',
    train_logits=train_logits_effnet,
    train_indices=train_indices_effnet,
    train_labels=train_labels_effnet,
    val_logits=val_logits_effnet,
    val_indices=val_indices_effnet,
    val_labels=val_labels_effnet
)

print(" Все сохранено в 'efficientnet_logits.npz'")

 Получение логитов от EfficientNetV2-S...
 DataLoader созданы:
   Train: 4906 батчей
   Val: 1052 батчей

 Получаем логиты для тренировочной выборки...


Получение логитов:   0%|          | 0/4906 [00:00<?, ?it/s]


 Получаем логиты для валидационной выборки...


Получение логитов:   0%|          | 0/1052 [00:00<?, ?it/s]


 Логиты получены:
   Train logits: (78481, 15)
   Val logits: (16821, 15)
   Train labels: (78481, 15)
   Val labels: (16821, 15)

 Сохраняем логиты...
 Все сохранено в 'efficientnet_logits.npz'


# загружаем логиты модели EfficientNet в переменные

In [6]:
import numpy as np

# Загружаем логиты EfficientNet
effnet_data = np.load('efficientnet_logits.npz')

# Записываем в переменные
train_logits_effnet = effnet_data['train_logits']
train_indices_effnet = effnet_data['train_indices']
train_labels_effnet = effnet_data['train_labels']

val_logits_effnet = effnet_data['val_logits']
val_indices_effnet = effnet_data['val_indices']
val_labels_effnet = effnet_data['val_labels']

print(" Логиты EfficientNet загружены")
print(f"Train: {train_logits_effnet.shape}")
print(f"Val: {val_logits_effnet.shape}")

✅ Логиты EfficientNet загружены
Train: (78481, 15)
Val: (16821, 15)


In [37]:
val_labels_effnet

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [42]:
np.max(train_logits_effnet)

1.3579513

In [46]:
lst = []

for i in train_logits_effnet:
    for j in i:
        if j < -5:
            lst.append(j)

In [1]:
len(lst)

NameError: name 'lst' is not defined

In [48]:
len(lst)

601

In [56]:
a = 1 / (1 + np.e** -(-40))

In [57]:
a

4.248354255291598e-18

# Загрузка модели Swin-Tiny

In [58]:
import torch
import torch.nn as nn
from torchvision import models, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

diseases_list = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 
                 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration',
                 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening',
                 'Pneumonia', 'Pneumothorax']

def get_swin_tiny(num_classes=15):
    model = models.swin_t(pretrained=False)
    in_features = model.head.in_features
    model.head = nn.Sequential(
        nn.Dropout(0.4, inplace=True),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.GELU(),
        nn.Dropout(0.2, inplace=True),
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.GELU(),
        nn.Dropout(0.133, inplace=True),
        nn.Linear(256, num_classes)
    )
    return model

swin_model = get_swin_tiny(num_classes=len(diseases_list))
checkpoint = torch.load('swin_tiny_final.pth', map_location=device)
swin_model.load_state_dict(checkpoint['model_state_dict'])
swin_model = swin_model.to(device)
swin_model.eval()

swin_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print(f" Swin-Tiny загружена на {device}")

C:\python\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\python\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


✅ Swin-Tiny загружена на cuda


# получение логитов Swin-Tiny

In [59]:
import numpy as np
from tqdm.notebook import tqdm
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class SwinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['full_path']
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, idx

# Создаем Dataset
meta_train_df = pd.read_csv('meta_train_data.csv')
meta_val_df = pd.read_csv('meta_val_data.csv')

train_dataset_swin = SwinDataset(meta_train_df, transform=swin_transform)
val_dataset_swin = SwinDataset(meta_val_df, transform=swin_transform)

# Создаем DataLoader
batch_size = 16
train_loader_swin = DataLoader(
    train_dataset_swin, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=0
)
val_loader_swin = DataLoader(
    val_dataset_swin, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=0
)

print(f" DataLoader созданы:")
print(f"   Train: {len(train_loader_swin)} батчей")
print(f"   Val: {len(val_loader_swin)} батчей")

# Функция получения логитов
def get_logits(model, loader, device):
    model.eval()
    all_logits = []
    all_indices = []
    
    with torch.no_grad():
        for images, indices in tqdm(loader, desc="Получение логитов"):
            images = images.to(device)
            logits = model(images)
            
            all_logits.append(logits.cpu().numpy())
            all_indices.append(indices.numpy())
    
    all_logits = np.vstack(all_logits)
    all_indices = np.concatenate(all_indices)
    
    return all_logits, all_indices

# Получаем логиты
print("\n Получаем логиты для тренировочной выборки...")
train_logits_swin, train_indices_swin = get_logits(swin_model, train_loader_swin, device)

print("\n Получаем логиты для валидационной выборки...")
val_logits_swin, val_indices_swin = get_logits(swin_model, val_loader_swin, device)

print(f"\n Логиты получены:")
print(f"   Train logits: {train_logits_swin.shape}")
print(f"   Val logits: {val_logits_swin.shape}")

# Получаем метки
train_labels_swin = meta_train_df.iloc[train_indices_swin][diseases_list].values.astype(np.float32)
val_labels_swin = meta_val_df.iloc[val_indices_swin][diseases_list].values.astype(np.float32)

print(f"   Train labels: {train_labels_swin.shape}")
print(f"   Val labels: {val_labels_swin.shape}")

# Сохраняем
print("\n Сохраняем логиты...")
np.savez_compressed(
    'swin_logits.npz',
    train_logits=train_logits_swin,
    train_indices=train_indices_swin,
    train_labels=train_labels_swin,
    val_logits=val_logits_swin,
    val_indices=val_indices_swin,
    val_labels=val_labels_swin
)

print(" Все сохранено в 'swin_logits.npz'")

 DataLoader созданы:
   Train: 4906 батчей
   Val: 1052 батчей

 Получаем логиты для тренировочной выборки...


Получение логитов:   0%|          | 0/4906 [00:00<?, ?it/s]


 Получаем логиты для валидационной выборки...


Получение логитов:   0%|          | 0/1052 [00:00<?, ?it/s]


 Логиты получены:
   Train logits: (78481, 15)
   Val logits: (16821, 15)
   Train labels: (78481, 15)
   Val labels: (16821, 15)

 Сохраняем логиты...
 Все сохранено в 'swin_logits.npz'


# загружаем логиты Swin-Tiny в переменные

In [1]:
import numpy as np

# Загружаем логиты Swin-Tiny
swin_data = np.load('swin_logits.npz')

# Записываем в переменные
train_logits_swin = swin_data['train_logits']
train_indices_swin = swin_data['train_indices']
train_labels_swin = swin_data['train_labels']

val_logits_swin = swin_data['val_logits']
val_indices_swin = swin_data['val_indices']
val_labels_swin = swin_data['val_labels']

print(" Логиты Swin-Tiny загружены")
print(f"Train: {train_logits_swin.shape}")
print(f"Val: {val_logits_swin.shape}")

 Логиты Swin-Tiny загружены
Train: (78481, 15)
Val: (16821, 15)


In [8]:
print(train_logits_effnet[0])
print(train_logits_dn121[0])
print(train_logits_swin[0])

[-1.1421154  -2.4482744  -1.6396525  -2.772853   -1.6476886  -1.4015819
 -1.2193284  -2.1206048  -0.5103397  -1.4912801   0.26581132 -1.0131999
 -1.4686044  -1.7722534  -1.7239482 ]
[-0.7233403  -1.268303   -1.1362191  -1.7943183  -0.79232186 -1.5521613
 -0.9515892  -0.9639831  -0.36645877 -0.84696835 -0.3473431  -0.742476
 -0.8106119  -1.3115737  -1.206772  ]
[-0.86213654 -2.0790489  -1.3186775  -2.1501486  -1.164455   -1.0898387
 -0.9638416  -1.7574348  -0.525435   -1.152984    0.11888577 -0.844658
 -0.9158355  -1.505312   -1.0224909 ]
